<a href="https://colab.research.google.com/github/AmrBr/Reverse-Dictionary/blob/main/Reverse_Dictionary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**INSTALLS**

In [2]:
!pip install camel-tools

**CONSTANTS**

In [20]:
SEED = 42
SPLITS = ["train", "validation", "test"]
COLUMNS_TO_REMOVE = ['id', 'pos', 'electra', 'bertseg', 'bertmsa']
WORD = 'word'
GLOSS = 'gloss'
DS1_PATH = 'Abreekaa/Arabic-Reverse-Dictionary'
DS2_PATH = 'riotu-lab/arabic_reverse_dictionary'

## Data Preprocessing



In [18]:
from datasets import load_dataset

def load_data():
  dataset_1 = load_dataset(DS1_PATH)
  dataset_bulk = load_dataset(DS2_PATH)

  dataset_bulk = dataset_bulk['train']
  return dataset_1, dataset_bulk

In [5]:
def remove_dublicates(ds1, ds2):
  seen = set()

  for split in SPLITS:
      for ex in ds1[split]:
          seen.add((ex[WORD], ex[GLOSS]))

  ds_bulk_clean = ds2.filter(lambda ex: not_duplicate(ex, seen))
  return ds_bulk_clean

def not_duplicate(ex, seen):
    return (ex[WORD], ex[GLOSS]) not in seen

In [12]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_maksura_ar, normalize_alef_ar, normalize_teh_marbuta_ar

def dediacritize(ex):
  return dediac_ar(ex).replace("ـ", "")

def normalize(ex):
  ex = normalize_alef_maksura_ar(ex)
  ex = normalize_alef_ar(ex)
  ex = normalize_teh_marbuta_ar(ex)
  return ex

In [16]:
from datasets import concatenate_datasets

def merge_datasets(ds1, ds2):
  return concatenate_datasets(ds1, ds2)

In [19]:
ds1, ds2 = load_data()

for split in SPLITS:
  ds1[split] = ds1[split].remove_columns(COLUMNS_TO_REMOVE)

  ds1[split] = ds1[split].map(lambda ex: {WORD: dediacritize(normalize(ex[WORD])), GLOSS: dediacritize(normalize(ex[GLOSS]))})


ds2 = ds2.rename_column("definition", "gloss")

ds2 = remove_dublicates(ds1, ds2)

ds2 = ds2.map(lambda ex: {WORD: dediacritize(normalize(ex[WORD])), GLOSS: dediacritize(normalize(ex[GLOSS]))})

train_ds = merge_datasets(ds1['train'], ds2)
validation_ds = ds1['validation']
test_ds = ds1['test']

Map:   0%|          | 0/31372 [00:00<?, ? examples/s]

Map:   0%|          | 0/3921 [00:00<?, ? examples/s]

Map:   0%|          | 0/3922 [00:00<?, ? examples/s]

Filter:   0%|          | 0/58607 [00:00<?, ? examples/s]

Map:   0%|          | 0/58607 [00:00<?, ? examples/s]

AttributeError: 'NoneType' object has no attribute 'replace'

In [ ]:
print(len(train_ds))
print(len(validation_ds))
print(len(test_ds))

In [ ]:
train_merged = train_ds.shuffle(seed=42)

In [ ]:
from datasets import DatasetDict

final_ds = DatasetDict({
    "train": train_merged,
    "validation": validation_ds,
    "test": test_ds
})

final_ds.push_to_hub("Abreekaa/arabic-reverse-dictionary-merged")